# IFRS S1/S2 report generation — clean LangGraph engine

A from-scratch, layered rewrite of the section-generation pipeline. It **reuses the old notebook only
for inputs**: run that notebook's prep phase once to produce the per-section artifacts
(`evidence_map_*`, `coverage_matrix_*`, `disclosure_plan_*`) under its `OUTPUT_DIR`; this engine loads
them and does the generation.

**Layers:** settings → typed models → io (prep loader) → llm → tools → claims → deterministic gates →
structured judges → deterministic approval → tool-using agents → LangGraph loop → assembly.

**Design:** no monkeypatching or global override chains; deterministic gates + approval are the binding
audit backbone (pure code); every LLM step is behind a protocol; writer/reviser are transparent
tool-using agents whose every tool call is logged. `pip install langgraph langchain-core langchain-openai`.

In [ ]:
import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Protocol, TypedDict

from langgraph.graph import StateGraph, END

## Settings

In [ ]:
@dataclass(frozen=True)
class Thresholds:
    judge_min: float = 7.0           # each judge must score >= this
    max_revisions: int = 3           # revision attempts before human review
    min_section_words: int = 120     # structural floor


@dataclass(frozen=True)
class Settings:
    output_dir: Path
    thresholds: Thresholds = field(default_factory=Thresholds)
    writer_max_steps: int = 6
    reviser_max_steps: int = 6

    @property
    def dirs(self) -> Dict[str, Path]:
        d = {name: self.output_dir / sub for name, sub in {
            "evidence_maps": "01_evidence_maps", "coverage": "02_coverage",
            "missing_requirements": "03_missing_requirements", "plans": "04_disclosure_plans",
            "claims": "06_claims_registers", "gates": "07_deterministic_gates",
            "judges": "08_judge_results", "revisions": "09_revised_sections",
            "approved": "10_approved_sections", "handoff": "12_pdf_handoff",
            "audit_logs": "audit_logs"}.items()}
        return d


SECTION_SLUGS = {
    "General Requirements": "general_requirements", "Governance": "governance",
    "Strategy": "strategy", "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}
SECTIONS = list(SECTION_SLUGS)

## Domain models (typed)

In [ ]:
@dataclass(frozen=True)
class EvidenceItem:
    id: str
    text: str
    value: Optional[str] = None

    def haystack(self) -> str:
        return f"{self.text} {self.value or ''}".lower()


@dataclass(frozen=True)
class Requirement:
    id: str
    text: str
    coverage: str = "unknown"        # covered | partial | not_available


@dataclass(frozen=True)
class SectionContext:
    """Immutable inputs for one section, loaded from prep artifacts."""
    section_name: str
    slug: str
    requirements: List[Requirement]
    evidence: List[EvidenceItem]
    disclosure_plan: Dict[str, Any]

    def supported_requirements(self) -> List[Requirement]:
        return [r for r in self.requirements if r.coverage in ("covered", "partial")]


@dataclass(frozen=True)
class Claim:
    text: str
    value: str
    evidence_id: Optional[str]       # None => unsupported

    @property
    def supported(self) -> bool:
        return self.evidence_id is not None


@dataclass(frozen=True)
class GateFinding:
    gate: str
    passed: bool
    issues: List[str] = field(default_factory=list)


@dataclass
class GateReport:
    findings: List[GateFinding]

    @property
    def passed(self) -> bool:
        return all(f.passed for f in self.findings)

    def failures(self) -> List[Dict[str, Any]]:
        return [{"gate": f.gate, "issues": f.issues} for f in self.findings if not f.passed]


@dataclass(frozen=True)
class JudgeScore:
    kind: str
    score: float
    issues: List[str] = field(default_factory=list)


@dataclass
class JudgeReport:
    scores: List[JudgeScore]

    def min_score(self) -> float:
        return min((s.score for s in self.scores), default=0.0)

    def below(self, threshold: float) -> List[JudgeScore]:
        return [s for s in self.scores if s.score < threshold]


@dataclass
class Approval:
    approved: bool
    failures: List[Dict[str, Any]]
    overall_score: float

## IO — load prep artifacts

In [ ]:
class ContextLoader:
    """Loads prep artifacts produced by the existing prep phase into typed SectionContext.

    Field names are defensive: prep JSON is read leniently so schema drift degrades to
    empty lists rather than crashes. Adjust the small `_extract_*` helpers if your prep
    uses different keys."""

    def __init__(self, settings: Settings):
        self.s = settings

    def _read(self, path: Path) -> Any:
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

    def load(self, section_name: str) -> SectionContext:
        slug = SECTION_SLUGS[section_name]
        coverage = self._read(self.s.dirs["coverage"] / f"coverage_matrix_{slug}.json") or {}
        evmap = self._read(self.s.dirs["evidence_maps"] / f"evidence_map_{slug}.json") or {}
        plan = self._read(self.s.dirs["plans"] / f"disclosure_plan_{slug}.json") or {}
        return SectionContext(
            section_name=section_name, slug=slug,
            requirements=self._extract_requirements(coverage),
            evidence=self._extract_evidence(evmap),
            disclosure_plan=plan if isinstance(plan, dict) else {"plan": plan},
        )

    @staticmethod
    def _extract_requirements(coverage: Any) -> List[Requirement]:
        rows = coverage.get("requirements", coverage) if isinstance(coverage, dict) else coverage
        out: List[Requirement] = []
        for r in (rows or []):
            if isinstance(r, dict):
                out.append(Requirement(id=str(r.get("id", r.get("requirement_id", ""))),
                                        text=str(r.get("text", r.get("requirement", ""))),
                                        coverage=str(r.get("coverage", r.get("status", "unknown"))).lower()))
        return out

    @staticmethod
    def _extract_evidence(evmap: Any) -> List[EvidenceItem]:
        items = evmap.get("evidence", evmap.get("items", evmap)) if isinstance(evmap, dict) else evmap
        out: List[EvidenceItem] = []
        for i, e in enumerate(items or []):
            if isinstance(e, dict):
                out.append(EvidenceItem(id=str(e.get("id", f"E{i}")),
                                        text=str(e.get("text", e.get("value", ""))),
                                        value=(str(e["value"]) if "value" in e else None)))
            elif isinstance(e, str):
                out.append(EvidenceItem(id=f"E{i}", text=e))
        return out

## LLM abstraction

In [ ]:
@dataclass
class ToolCall:
    name: str
    args: Dict[str, Any]
    id: str = ""


@dataclass
class AIResponse:
    content: str = ""
    tool_calls: List[ToolCall] = field(default_factory=list)


class ChatModel(Protocol):
    def complete(self, messages: List[Dict[str, Any]], tools: Optional[List["Tool"]] = None) -> AIResponse: ...


class LangChainChatModel:
    """Adapter over a LangChain chat model with .bind_tools (e.g. AzureChatOpenAI)."""
    def __init__(self, lc_model): self._m = lc_model

    def complete(self, messages, tools=None):
        from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
        conv = []
        for m in messages:
            r = m["role"]
            if r == "system": conv.append(SystemMessage(m["content"]))
            elif r == "user": conv.append(HumanMessage(m["content"]))
            elif r == "assistant": conv.append(AIMessage(m.get("content", ""), tool_calls=m.get("tool_calls", [])))
            elif r == "tool": conv.append(ToolMessage(m["content"], tool_call_id=m.get("tool_call_id", "")))
        model = self._m.bind_tools([t.as_lc() for t in tools]) if tools else self._m
        ai = model.invoke(conv)
        tcs = [ToolCall(t["name"], t.get("args", {}), t.get("id", "")) for t in getattr(ai, "tool_calls", []) or []]
        return AIResponse(content=ai.content or "", tool_calls=tcs)


def parse_json_object(text: str) -> Optional[dict]:
    try:
        return json.loads(text[text.index("{"): text.rindex("}") + 1])
    except Exception:
        return None

## Tools + bounded agent loop

In [ ]:
@dataclass
class Tool:
    name: str
    description: str
    func: Callable[..., Any]

    def as_lc(self):
        from langchain_core.tools import StructuredTool
        return StructuredTool.from_function(func=self.func, name=self.name, description=self.description)


def retrieval_tools(ctx: SectionContext) -> List[Tool]:
    return [
        Tool("get_requirements", "IFRS requirements assigned to this section (id, text, coverage).",
             lambda: json.dumps([r.__dict__ for r in ctx.supported_requirements()])[:6000]),
        Tool("get_evidence", "BANK01 evidence available for this section (id, text, value).",
             lambda: json.dumps([e.__dict__ for e in ctx.evidence])[:9000]),
        Tool("get_disclosure_plan", "The disclosure plan/blueprint for this section.",
             lambda: json.dumps(ctx.disclosure_plan)[:6000]),
        Tool("search_evidence", "Search this section's evidence for a keyword or figure.",
             lambda query: json.dumps([e.__dict__ for e in ctx.evidence
                                       if str(query).lower() in e.haystack()][:20])[:9000]),
    ]


class ToolAgent:
    """Bounded, fully-logged tool loop. Returns (final_text, trace)."""
    def __init__(self, model: ChatModel, tools: List[Tool], max_steps: int):
        self.model, self.tools = model, tools
        self.reg = {t.name: t for t in tools}
        self.max_steps = max_steps

    def run(self, system: str, user: str):
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        trace: List[Dict[str, Any]] = []
        for _ in range(self.max_steps):
            resp = self.model.complete(messages, tools=self.tools)
            if not resp.tool_calls:
                return resp.content, trace
            messages.append({"role": "assistant", "content": resp.content,
                             "tool_calls": [tc.__dict__ for tc in resp.tool_calls]})
            for tc in resp.tool_calls:
                try:
                    obs = self.reg[tc.name].func(**tc.args)
                except Exception as e:
                    obs = f"ERROR: {e}"
                trace.append({"tool": tc.name, "args": tc.args})
                messages.append({"role": "tool", "content": str(obs)[:12000], "tool_call_id": tc.id})
        final = self.model.complete(messages + [{"role": "user", "content": "Provide the final Markdown now."}])
        return final.content, trace

## Claims (deterministic)

In [ ]:
_NUMERIC = re.compile(
    r"(?<![\w.])(?:[€$£]\s?)?\d+(?:,\d{3})*(?:\.\d+)?"
    r"(?:\s?(?:tCO2e|tCO\u2082e|kt|MWh|GWh|EUR|USD|years?|%|t)\b)?", re.I)


def build_claims(draft: str, ctx: SectionContext) -> List[Claim]:
    """Extract quantitative claims (numbers/units/years) and map each to an evidence item
    by value overlap. A claim with no matching evidence is 'unsupported' -> fails grounding."""
    claims: List[Claim] = []
    for m in _NUMERIC.finditer(draft):
        value = m.group(0).strip().rstrip(",.;:")
        if len(re.sub(r"[^\d]", "", value)) < 2:   # skip trivial single digits
            continue
        norm = re.sub(r"[,\s]", "", value).lower()
        ev = next((e for e in ctx.evidence if norm and norm in re.sub(r"[,\s]", "", e.haystack())), None)
        claims.append(Claim(text=draft[max(0, m.start() - 40):m.end() + 20].strip(),
                            value=value, evidence_id=ev.id if ev else None))
    return claims

## Deterministic gates

In [ ]:
_PLACEHOLDERS = re.compile(r"\[[^\]]*(insert|tbd|add|placeholder|xxx)[^\]]*\]|\bTBD\b|\bXXX\b", re.I)
_MISSING_LANG = re.compile(
    r"\b(not\s+(?:available|reported|disclosed|provided)|no\s+data|data\s+gap|unavailable|"
    r"not\s+applicable\s+data|absence\s+of\s+data)\b", re.I)


def grounding_gate(claims: List[Claim]) -> GateFinding:
    unsupported = [c.value for c in claims if not c.supported]
    return GateFinding("grounding", not unsupported,
                       [f"claim not traceable to evidence: {v}" for v in unsupported[:10]])


def cleanliness_gate(draft: str) -> GateFinding:
    issues = []
    if _PLACEHOLDERS.search(draft):
        issues.append("contains template placeholders")
    if _MISSING_LANG.search(draft):
        issues.append("contains missing/unavailable-data language")
    return GateFinding("cleanliness", not issues, issues)


def structure_gate(draft: str, ctx: SectionContext, min_words: int) -> GateFinding:
    issues = []
    if len(draft.split()) < min_words:
        issues.append(f"below minimum length ({min_words} words)")
    if not re.search(r"^#{1,3}\s", draft, re.M):
        issues.append("no Markdown headings")
    return GateFinding("structure", not issues, issues)


def claims_integrity_gate(claims: List[Claim]) -> GateFinding:
    # every extracted claim must have a value; supported ones must reference evidence
    issues = [f"malformed claim: {c.text[:40]}" for c in claims if not c.value]
    return GateFinding("claims_integrity", not issues, issues)


def run_gates(draft: str, ctx: SectionContext, settings: Settings) -> tuple[GateReport, List[Claim]]:
    claims = build_claims(draft, ctx)
    report = GateReport([
        grounding_gate(claims),
        cleanliness_gate(draft),
        structure_gate(draft, ctx, settings.thresholds.min_section_words),
        claims_integrity_gate(claims),
    ])
    return report, claims

## Structured judges

In [ ]:
_JUDGE_PROMPTS = {
    "coverage": ("Score how completely the section addresses its assigned IFRS requirements. "
                 "Penalise omitted mandatory disclosures."),
    "evidence": ("Score how well every quantitative and factual claim is supported by the provided "
                 "evidence. Penalise any claim not traceable to evidence."),
    "style": ("Score IFRS-appropriate tone, clarity and neutrality. Penalise marketing language, "
              "hedging, placeholders, or references to missing data."),
}


def run_judges(model: ChatModel, draft: str, ctx: SectionContext) -> JudgeReport:
    evidence = json.dumps([e.__dict__ for e in ctx.evidence])[:6000]
    reqs = json.dumps([r.__dict__ for r in ctx.supported_requirements()])[:4000]
    scores: List[JudgeScore] = []
    for kind, rubric in _JUDGE_PROMPTS.items():
        system = (f"You are an IFRS S1/S2 {kind} judge. {rubric} "
                  f'Return ONLY JSON: {{"score_0_to_10": <number>, "issues": [<strings>]}}.')
        user = f"REQUIREMENTS:\n{reqs}\n\nEVIDENCE:\n{evidence}\n\nSECTION:\n{draft}"
        parsed = parse_json_object(model.complete([{"role": "system", "content": system},
                                                   {"role": "user", "content": user}]).content) or {}
        scores.append(JudgeScore(kind=kind, score=float(parsed.get("score_0_to_10", 0.0)),
                                 issues=list(parsed.get("issues", []))))
    return JudgeReport(scores)

## Approval (single source of truth)

In [ ]:
def decide_approval(gates: GateReport, judges: Optional[JudgeReport], settings: Settings) -> Approval:
    failures = list(gates.failures())
    judge_min = settings.thresholds.judge_min
    if judges is not None:
        for js in judges.below(judge_min):
            failures.append({"gate": f"judge_{js.kind}", "issues": js.issues or [f"score {js.score} < {judge_min}"]})
    overall = judges.min_score() if judges is not None else 0.0
    approved = gates.passed and judges is not None and not judges.below(judge_min)
    return Approval(approved=approved, failures=failures, overall_score=overall)

## Writer + reviser agents

In [ ]:
WRITER_RULES = (
    "Write final-report IFRS prose grounded strictly in retrieved evidence. Use real values from "
    "get_evidence; never use placeholders, template rows, or instructions to the entity. Never write "
    "about missing/unavailable data; if evidence is partial, write only the supported subset. Do not "
    "invent committees, policies, targets, metrics, dates, currencies or figures. Every quantitative "
    "claim must be traceable to an evidence item. Use neutral IFRS-aligned language.")

WRITER_SYSTEM = ("You are an IFRS S1/S2 disclosure writer. First call get_requirements, get_evidence and "
                 "get_disclosure_plan; use search_evidence for specific figures. Then write the section in "
                 "Markdown with appropriate headings. " + WRITER_RULES)
REVISER_SYSTEM = ("You are an IFRS reviser. Read the listed failures, fetch evidence with the tools, and make "
                  "the minimal corrections needed. Return the corrected Markdown only. " + WRITER_RULES)


def write_draft(model: ChatModel, ctx: SectionContext, settings: Settings) -> tuple[str, list]:
    agent = ToolAgent(model, retrieval_tools(ctx), settings.writer_max_steps)
    return agent.run(WRITER_SYSTEM, f"Write the IFRS section '{ctx.section_name}'.")


def revise_draft(model: ChatModel, ctx: SectionContext, draft: str,
                 failures: List[Dict[str, Any]], settings: Settings) -> tuple[str, list]:
    agent = ToolAgent(model, retrieval_tools(ctx), settings.reviser_max_steps)
    user = f"Section '{ctx.section_name}' failed: {failures}\n\nCurrent draft:\n\n{draft}\n\nReturn corrected Markdown."
    return agent.run(REVISER_SYSTEM, user)

## LangGraph loop

In [ ]:
class SectionState(TypedDict, total=False):
    ctx: SectionContext
    draft: str
    claims: List[Claim]
    gates: GateReport
    judges: Optional[JudgeReport]
    approval: Approval
    iteration: int
    prev_failures: Any
    trace: List[Dict[str, Any]]
    result: Dict[str, Any]


def build_graph(model: ChatModel, settings: Settings):

    def n_generate(s: SectionState) -> SectionState:
        draft, trace = write_draft(model, s["ctx"], settings)
        return {"draft": draft, "iteration": 0, "prev_failures": None,
                "trace": [{"agent": "writer", "steps": trace}]}

    def n_evaluate(s: SectionState) -> SectionState:
        gates, claims = run_gates(s["draft"], s["ctx"], settings)
        return {"gates": gates, "claims": claims}

    def route_gates(s: SectionState) -> str:
        return "judge" if s["gates"].passed else "approve"

    def n_judge(s: SectionState) -> SectionState:
        return {"judges": run_judges(model, s["draft"], s["ctx"])}

    def n_approve(s: SectionState) -> SectionState:
        approval = decide_approval(s["gates"], s.get("judges"), settings)
        return {"approval": approval}

    def route_approve(s: SectionState) -> str:
        if s["approval"].approved:
            return "approved"
        cur = json.dumps(s["approval"].failures, sort_keys=True, default=str)
        if cur == s["prev_failures"] and s["gates"].passed:
            return "human_review"          # same issue persists -> escalate
        if s["iteration"] >= settings.thresholds.max_revisions:
            return "human_review"
        return "revise"

    def n_revise(s: SectionState) -> SectionState:
        draft, trace = revise_draft(model, s["ctx"], s["draft"], s["approval"].failures, settings)
        prev = json.dumps(s["approval"].failures, sort_keys=True, default=str)
        return {"draft": draft, "iteration": s["iteration"] + 1, "prev_failures": prev,
                "trace": [{"agent": "reviser", "iteration": s["iteration"], "steps": trace}]}

    def n_approved(s: SectionState) -> SectionState:
        return {"result": _finish(s, "approved", settings)}

    def n_human(s: SectionState) -> SectionState:
        return {"result": _finish(s, "human_review", settings)}

    g = StateGraph(SectionState)
    for name, fn in [("generate", n_generate), ("evaluate", n_evaluate), ("judge", n_judge),
                     ("approve", n_approve), ("revise", n_revise),
                     ("approved", n_approved), ("human_review", n_human)]:
        g.add_node(name, fn)
    g.set_entry_point("generate")
    g.add_edge("generate", "evaluate")
    g.add_conditional_edges("evaluate", route_gates, {"judge": "judge", "approve": "approve"})
    g.add_edge("judge", "approve")
    g.add_conditional_edges("approve", route_approve,
                            {"approved": "approved", "revise": "revise", "human_review": "human_review"})
    g.add_edge("revise", "evaluate")
    g.add_edge("approved", END)
    g.add_edge("human_review", END)
    return g.compile()


def _finish(s: SectionState, status: str, settings: Settings) -> Dict[str, Any]:
    ctx = s["ctx"]
    record = {
        "section_name": ctx.section_name, "slug": ctx.slug, "status": status,
        "iterations": s.get("iteration", 0), "markdown": s["draft"],
        "overall_score": s["approval"].overall_score if s.get("approval") else 0.0,
        "failures": s["approval"].failures if s.get("approval") else [],
        "claims": [c.__dict__ for c in s.get("claims", [])],
        "agent_trace": s.get("trace", []),
    }
    prefix = "approved" if status == "approved" else "human_review"
    _write_json(record, settings.dirs["approved"] / f"{prefix}_{ctx.slug}.json")
    _write_text(s["draft"], settings.dirs["approved"] / f"{prefix}_{ctx.slug}.md")
    return record

## Assembly + run

In [ ]:
def _write_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def run_section(section_name: str, loader: ContextLoader, model: ChatModel, settings: Settings) -> Dict[str, Any]:
    graph = build_graph(model, settings)
    limit = (settings.thresholds.max_revisions + 1) * 6 + 10
    out = graph.invoke({"ctx": loader.load(section_name)}, config={"recursion_limit": limit})
    return out["result"]


def run_report(loader: ContextLoader, model: ChatModel, settings: Settings,
               sections: Optional[List[str]] = None) -> Dict[str, Any]:
    results = [run_section(sn, loader, model, settings) for sn in (sections or SECTIONS)]
    report_md = assemble_report(results)
    _write_text(report_md, settings.dirs["handoff"] / "approved_report.md")
    _write_json({"sections": results, "approved": sum(r["status"] == "approved" for r in results),
                 "total": len(results)}, settings.dirs["audit_logs"] / "run_summary.json")
    return {"results": results, "report_markdown": report_md}


def assemble_report(results: List[Dict[str, Any]]) -> str:
    parts = ["# IFRS S1/S2 Sustainability Report\n"]
    for r in results:
        flag = "" if r["status"] == "approved" else "  _(pending human review)_"
        parts.append(f"\n## {r['section_name']}{flag}\n\n{r['markdown'].strip()}\n")
    return "\n".join(parts)

## Self-contained smoke test (no Azure, no data)

In [ ]:
def _clean_engine_smoke_test():
    import json, tempfile
    from pathlib import Path
    tmp = Path(tempfile.mkdtemp()); st = Settings(output_dir=tmp); d = st.dirs; slug = "governance"
    for k in ("coverage", "evidence_maps", "plans"): d[k].mkdir(parents=True, exist_ok=True)
    json.dump({"requirements": [{"id": "G1", "text": "Board oversight of climate risks", "coverage": "covered"}]},
              open(d["coverage"] / f"coverage_matrix_{slug}.json", "w"))
    json.dump({"evidence": [{"id": "E1", "text": "Board committee met four times in 2024", "value": "2024"},
                            {"id": "E3", "text": "Emissions reduced by 1250 tCO2e", "value": "1250 tCO2e"}]},
              open(d["evidence_maps"] / f"evidence_map_{slug}.json", "w"))
    json.dump({"subsections": ["Board oversight", "Management role"]},
              open(d["plans"] / f"disclosure_plan_{slug}.json", "w"))

    GOOD = """## Board oversight

The board maintains oversight of climate-related risks and opportunities through its dedicated sustainability committee.
During the reporting period ending in 2024, the committee reviewed the entity's climate strategy, its transition plan, and
progress against approved targets, and considered management's assessment of climate-related risks across the organisation.
The committee ensures climate considerations are integrated into the entity's overall risk governance framework and reports
to the full board regularly.

## Management role

Day-to-day responsibility rests with senior management, led by the Chief Sustainability Officer appointed in 2024.
Management monitors climate metrics, oversees implementation of climate policies, and tracks operational performance,
including a reduction of 1250 tCO2e over the period, reporting regularly to the board committee on the delivery of the
entity's climate commitments and transition plan across principal units."""

    BAD = "## Board oversight\n\nThe board met [INSERT NUMBER] times and cut emissions by 999 tCO2e. Data not available for targets."

    class FakeModel:
        def complete(self, messages, tools=None):
            if "judge" in messages[0]["content"].lower():
                return AIResponse(content='{"score_0_to_10": 8, "issues": []}')
            if tools and not any(m["role"] == "tool" for m in messages):
                return AIResponse(tool_calls=[ToolCall("get_evidence", {}, "e")])
            return AIResponse(content=GOOD)

    class BadModel(FakeModel):
        def complete(self, messages, tools=None):
            if "judge" in messages[0]["content"].lower():
                return AIResponse(content='{"score_0_to_10": 8}')
            if tools and not any(m["role"] == "tool" for m in messages):
                return AIResponse(tool_calls=[ToolCall("get_evidence", {}, "e")])
            return AIResponse(content=BAD)

    r = run_report(ContextLoader(st), FakeModel(), st, sections=["Governance"])["results"][0]
    assert r["status"] == "approved" and all(c["evidence_id"] for c in r["claims"]), r
    print("positive: approved @ iter", r["iterations"], "| score", r["overall_score"],
          "| claims", [(c["value"], c["evidence_id"]) for c in r["claims"]])

    r2 = run_report(ContextLoader(st), BadModel(), st, sections=["Governance"])["results"][0]
    assert r2["status"] == "human_review", r2
    print("negative: held for review | failing gates:", sorted({f["gate"] for f in r2["failures"]}))
    print("CLEAN ENGINE VERIFIED.")


_clean_engine_smoke_test()


## Production run (Azure + your prep output)

In [ ]:
import os
from pathlib import Path

# Point at the SAME directory your prep phase wrote to (the old notebook's OUTPUT_DIR).
PREP_OUTPUT_DIR = Path(os.environ.get("GENERATION_OUTPUT_DIR", "./generated_reports/agentic_ifrs_report")).resolve()
settings = Settings(output_dir=PREP_OUTPUT_DIR)

def make_azure_model():
    """Tool-calling Azure model. The deployment must support function/tool calling."""
    from langchain_openai import AzureChatOpenAI
    return LangChainChatModel(AzureChatOpenAI(
        azure_deployment=os.environ.get("AZURE_AGENT_DEPLOYMENT", os.environ.get("AZURE_OPENAI_DEPLOYMENT", "")),
        api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
        temperature=0))

# result = run_report(ContextLoader(settings), make_azure_model(), settings)
# print("approved:", sum(r["status"]=="approved" for r in result["results"]), "/", len(result["results"]))
# print(result["report_markdown"][:2000])


### How to run
1. Run the **old notebook's prep phase** once to produce `evidence_map_*`, `coverage_matrix_*`,
   `disclosure_plan_*` under its `OUTPUT_DIR`.
2. `pip install langgraph langchain-core langchain-openai`; run every engine cell above.
3. Run the **smoke test** cell to confirm the graph, gates, judges and approval work (no Azure needed).
4. Set `GENERATION_OUTPUT_DIR` to that prep output dir and your Azure env vars, then run
   `result = run_report(ContextLoader(settings), make_azure_model(), settings)`.

**Validation (since generation is agentic):** compare the approved sections against your current
reports; tune `Thresholds`, the writer/judge prompts, and the claim/gate patterns. The `ContextLoader`
`_extract_*` helpers read prep JSON leniently — adjust their keys if your prep schema differs.